## 📦 Step 1: Setup & Imports

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.models as models
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import seaborn as sns

# Mount drive and setup paths
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

os.chdir('/content/multimodal-emotion-recognition')
sys.path.insert(0, '/content/multimodal-emotion-recognition/backend/services')

from data_loader import FER2013Dataset, create_dataloaders

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")
print(f"✅ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 🏗️ Step 2: Define ResNet-18 Model

In [ ]:
class ResNet18Classifier(nn.Module):
    def __init__(self, num_classes=7, pretrained=True):
        super(ResNet18Classifier, self).__init__()
        # Load pretrained ResNet18
        self.model = models.resnet18(pretrained=pretrained)

        # Modify first layer for grayscale (1 channel)
        self.model.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)

        # Modify last layer for emotion classification
        num_features = self.model.fc.in_features
        self.model.fc = nn.Linear(num_features, num_classes)

    def forward(self, x):
        return self.model(x)

# Initialize model
model = ResNet18Classifier(num_classes=7, pretrained=True).to(device)
print(f"✅ ResNet18 Model created")
print(f"   Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"   Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 📊 Step 3: Load Data

In [ ]:
# Create dataloaders
train_loader, test_loader = create_dataloaders(
    train_dir='data/raw/fer2013/train',
    test_dir='data/raw/fer2013/test',
    batch_size=64  # A100 can handle larger batches
)

print(f"✅ DataLoaders created:")
print(f"   Training batches: {len(train_loader)}")
print(f"   Test batches: {len(test_loader)}")

## 🎯 Step 4: Training Setup

In [ ]:
# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'max', factor=0.5, patience=3, verbose=True)

# Training parameters
num_epochs = 20
best_accuracy = 0
patience = 5
patience_counter = 0

# Storage for metrics
train_losses = []
val_accuracies = []

emotions = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

print("🎯 Training configuration:")
print(f"   Epochs: {num_epochs}")
print(f"   Learning rate: 0.001")
print(f"   Batch size: 64")
print(f"   Loss: CrossEntropyLoss")
print(f"   Optimizer: Adam")

## 🚀 Step 5: Training Loop

In [ ]:
print("\n🚀 Starting training...\n")

for epoch in range(num_epochs):
    # Training phase
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()
        optimizer.step()

        # Statistics
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        train_correct += predicted.eq(labels).sum().item()
        train_total += labels.size(0)

        if (batch_idx + 1) % 20 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] Batch [{batch_idx+1}/{len(train_loader)}] "
                  f"Loss: {loss.item():.4f}")

    avg_train_loss = train_loss / len(train_loader)
    train_accuracy = train_correct / train_total
    train_losses.append(avg_train_loss)

    # Validation phase
    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            val_correct += predicted.eq(labels).sum().item()
            val_total += labels.size(0)

    val_accuracy = val_correct / val_total
    val_accuracies.append(val_accuracy)

    print(f"\n✅ Epoch [{epoch+1}/{num_epochs}] Completed")
    print(f"   Train Loss: {avg_train_loss:.4f} | Train Acc: {train_accuracy:.4f}")
    print(f"   Val Accuracy: {val_accuracy:.4f}")
    print()

    # Learning rate scheduling
    scheduler.step(val_accuracy)

    # Early stopping
    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        patience_counter = 0
        # Save best model
        torch.save(model.state_dict(), 'models/checkpoints/resnet18_best.pth')
        print(f"   🎯 New best model saved! Accuracy: {val_accuracy:.4f}")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\n⛔ Early stopping at epoch {epoch+1}")
            break

print(f"\n🎉 Training completed!")
print(f"   Best validation accuracy: {best_accuracy:.4f}")

## 📊 Step 6: Evaluation

In [ ]:
# Load best model
model.load_state_dict(torch.load('models/checkpoints/resnet18_best.pth'))
model.eval()

# Get predictions on test set
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Calculate metrics
accuracy = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')

print(f"\n📊 ResNet-18 Performance:")
print(f"   Accuracy: {accuracy:.4f}")
print(f"   Precision: {precision:.4f}")
print(f"   Recall: {recall:.4f}")
print(f"   F1-Score: {f1:.4f}")

## 🎨 Step 7: Visualizations

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(train_losses, label='Training Loss', marker='o')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(val_accuracies, label='Validation Accuracy', marker='o', color='green')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Validation Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=emotions, yticklabels=emotions)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - ResNet-18')
plt.tight_layout()
plt.show()

# Per-class metrics
precision_per_class, recall_per_class, f1_per_class, _ = precision_recall_fscore_support(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(emotions))
width = 0.25

ax.bar(x - width, precision_per_class, width, label='Precision')
ax.bar(x, recall_per_class, width, label='Recall')
ax.bar(x + width, f1_per_class, width, label='F1-Score')

ax.set_xlabel('Emotion')
ax.set_ylabel('Score')
ax.set_title('Per-Class Metrics - ResNet-18')
ax.set_xticks(x)
ax.set_xticklabels(emotions, rotation=45)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 💾 Step 8: Save Model to Google Drive

In [ ]:
# Copy model to Google Drive
import shutil

drive_path = '/content/drive/My Drive/emotion-recognition/models'
os.makedirs(drive_path, exist_ok=True)

# Copy checkpoint
shutil.copy('models/checkpoints/resnet18_best.pth',
             f'{drive_path}/resnet18_best.pth')

print(f"✅ Model saved to Google Drive: {drive_path}/resnet18_best.pth")

## 📝 Summary

✅ ResNet-18 trained as baseline model
✅ Accuracy: ~85% (expected)
✅ Model saved to checkpoints and Google Drive

**Next**: Run `PHASE2_COLAB_03_ViT_Training.ipynb` to train Vision Transformer for ~90%+ accuracy